In [ ]:
import json
import os
import time
import glob

import pandas as pd
import plotly.express as px
import requests
from src.opensky_auth import tokens
import src.geo as geo

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

e:\Projects\Git\flight-trajectory-prediction


In [ ]:
STATE_COLUMNS = [
		"icao24",
		"callsign",
		"origin_country",
		"time_position",
		"last_contact",
		"longitude",
		"latitude",
		"baro_altitude",
		"on_ground",
		"velocity",
		"true_track",
		"vertical_rate",
		"sensors",
		"geo_altitude",
		"squawk",
		"spi",
		"position_source",
		"category"
]

PARIS_BBOX = {
	"lamin": 48.5,
	"lomin": 1.8,
	"lamax": 49.5,
	"lomax": 3.3,
}

In [88]:
def jprint(obj):
    text = json.dumps(obj, sort_keys=True, indent=4)
    print(text)


In [89]:
def states_to_dataframe(data):
    """Convert an OpenSky /states/all response into a DataFrame with one row per aircraft.
    
    Columns follow STATE_COLUMNS plus snapshot_time: the response's
    timestamp as a UTC datetime, the same for every row.
    Returns an empty DataFrame with those columns
    if the bounding box contains no aircraft.
    """
    
    df = pd.DataFrame(data["states"], columns=STATE_COLUMNS)
    df["snapshot_time"] = pd.to_datetime(data["time"], unit="s", utc=True)
    return df 

In [90]:
def fetch_snapshot(lamin, lomin, lamax, lomax):
    """Fetch the current state of all aircraft inside a bounding box from OpenSky.
    
    Args:
        lamin, lamax: Minimum and maximum latitude, in degrees.
        lomin, lomax: Minimum and maximum longitude, in degrees.
        
    Returns a DataFrame with one row per aircraft (see states_to_dataframe).
    Raises an HTTPError if the request fails.
    """
    params = {
		"lamin": lamin,
		"lomin": lomin,
		"lamax": lamax,
		"lomax": lomax,
		"extended": 1
	}
    
    response = requests.get(
		"https://opensky-network.org/api/states/all",
		headers=tokens.headers(),
		params=params,
		timeout=10
	)
    
    response.raise_for_status()
    
    data = response.json()
    df = states_to_dataframe(data)
    
    return df


In [91]:
df = fetch_snapshot(**PARIS_BBOX)
print(df)

    icao24  callsign              origin_country  time_position  last_contact  \
0   39de4d  TVF18BM                       France     1794931716    1794931716   
1   39de4c  TVF21YB                       France     1794931753    1794931753   
2   4952cf  TAP36G                      Portugal     1794931753    1794931753   
3   511195  CFG5XN                       Estonia     1794931752    1794931752   
4   4400ec  EJU69MC                      Austria     1794931530    1794931530   
5   39e692  AFR19CE                       France     1794931752    1794931753   
6   4009c6  BAW742                United Kingdom     1794931753    1794931753   
7   73806e  ELY222                        Israel     1794931753    1794931753   
8   39dd45  CCM96VK                       France     1794931749    1794931749   
9   4bb1e1  THY3AB                        Turkey     1794931753    1794931753   
10  39dd04  TVF88QM                       France     1794931747    1794931751   
11  3451d3  VLG6RE          

In [92]:
n_total = len(df)
geo_null = df["geo_altitude"].isna()
baro_null = df["baro_altitude"].isna()

print(f"Aircrafts in snapshot: {n_total}")
print(f"geo_altitude missing:  {geo_null.sum()} ({geo_null.mean() * 100:.1f}%)")
print(f"baro_altitude missing: {baro_null.sum()} ({baro_null.mean() * 100:.1f}%)")
print(f"Both missing: {(geo_null & baro_null).sum()}")

print("\nOn ground among aircrafts")
print(f"missing geo_altitude: {df[geo_null]["on_ground"].sum()}")
print(f"missing baro_altitude: {df[baro_null]["on_ground"].sum()}")

airborne = df[~df["on_ground"]]
print(f"\nAirborne aircraft: {len(airborne)}")
print(f"geo_altitude missing: {airborne["geo_altitude"].isna().mean() * 100:.1f}%")
print(f"baro_altitude missing: {airborne["baro_altitude"].isna().mean() * 100:.1f}%")

Aircrafts in snapshot: 33
geo_altitude missing:  13 (39.4%)
baro_altitude missing: 13 (39.4%)
Both missing: 13

On ground among aircrafts
missing geo_altitude: 13
missing baro_altitude: 13

Airborne aircraft: 20
geo_altitude missing: 0.0%
baro_altitude missing: 0.0%


In [93]:
def prepare_map_data(df):
	"""Prepare state vectors for the live map.
	
	Keeps only airborne aircraft with a known position 
 	(where latitude and longitude are not missing),
	and adds a geo_altitude_ft column, converting geo_altitude from meters to feet.
	Returns a new DataFrame, the input data is not modified.
	"""
	new_df = df[~df["on_ground"]].copy()
	new_df = new_df.dropna(subset=["latitude", "longitude"])
	new_df["geo_altitude_ft"] = new_df["geo_altitude"] * 3.28084
	new_df["velocity_kt"] = new_df["velocity"] * 1.94384
	return new_df

In [94]:
DURATION = 15 * 60
INTERVAL = 15

os.makedirs("data/raw", exist_ok=True)

snapshots = []
start = time.time()
round_nb = 0

while time.time() - start < DURATION:
    round_nb += 1

    try:
        df = fetch_snapshot(**PARIS_BBOX)
        snapshots.append(df)
        print(
            f"Round {round_nb}: {len(df)} aircraft at {df['snapshot_time'].iloc[0]:%H:%M:%S} UTC"
            if len(df)
            else f"Round {round_nb}: no aircraft"
        )
    except requests.exceptions.RequestException as e:
        print(f"Round {round_nb}: failed to fetch snapshot: {e}")

    time.sleep(INTERVAL)

Round 1: 33 aircraft at 16:09:16 UTC
Round 2: 34 aircraft at 16:09:30 UTC
Round 3: 34 aircraft at 16:09:36 UTC
Round 4: 35 aircraft at 16:09:58 UTC
Round 5: 37 aircraft at 16:10:14 UTC
Round 6: 36 aircraft at 16:10:30 UTC
Round 7: 35 aircraft at 16:10:46 UTC
Round 8: 35 aircraft at 16:10:59 UTC
Round 9: 35 aircraft at 16:11:19 UTC
Round 10: 35 aircraft at 16:11:34 UTC
Round 11: 35 aircraft at 16:11:40 UTC
Round 12: 34 aircraft at 16:12:05 UTC
Round 13: 34 aircraft at 16:12:20 UTC
Round 14: 34 aircraft at 16:12:36 UTC
Round 15: 34 aircraft at 16:12:36 UTC
Round 16: 34 aircraft at 16:13:05 UTC
Round 17: 35 aircraft at 16:13:19 UTC
Round 18: 37 aircraft at 16:13:35 UTC
Round 19: 39 aircraft at 16:13:54 UTC
Round 20: 38 aircraft at 16:14:10 UTC
Round 21: 37 aircraft at 16:14:25 UTC
Round 22: 40 aircraft at 16:14:41 UTC
Round 23: 40 aircraft at 16:14:41 UTC
Round 24: 38 aircraft at 16:15:09 UTC
Round 25: 38 aircraft at 16:15:25 UTC
Round 26: 39 aircraft at 16:15:40 UTC
Round 27: 41 aircraft

In [ ]:
if snapshots:
    all_states = pd.concat(snapshots, ignore_index=True)
    
    start_label = all_states["snapshot_time"].min().strftime("%Y-%m-%d_%H-%M")
    path = f"data/raw/states_paris_{start_label}.parquet"
    
    all_states.to_parquet(path, index=False) 
    
    print(f"Saved {len(all_states)} rows from {len(snapshots)} snapshots to {path}")
else:
    print("No snapshots collected, nothing saved.")

Saved 2276 rows from 58 snapshots to data/raw/states_paris_2026-11-17_16-09.parquet


In [108]:
latest_states = max(glob.glob("data/raw/states_paris_*.parquet"))

snapshots = pd.read_parquet(latest_states)
snapshots = snapshots.sort_values(by=["icao24", "snapshot_time"])

In [ ]:
map_df = prepare_map_data(snapshots)

first = map_df["snapshot_time"].min()
last = map_df["snapshot_time"].max()

title = f"Aircraft tracks around Paris — {first:%d %b %Y, %H:%M}–{last:%H:%M} UTC"

fig = px.line_map(
	data_frame=map_df,
	lat="latitude",
	lon="longitude",
	hover_name="callsign",
	hover_data=["velocity_kt", "true_track", "geo_altitude_ft"],
	center={
		"lat": 48.86,
		"lon": 2.35
	},
	zoom=8,
	color="callsign",
 	map_style="carto-positron",
  	title=title
)

file_path = f"plots/tracks_paris_{first:%Y-%m-%d_%H-%M}.html"
fig.write_html(file_path)
fig.show()

In [ ]:
CDG_COORDS = {
	"lat": 49.01,
	"lon": 2.55
}

MAX_DISTANCE_KM = 35
MAX_ALTITUDE_M = 3000

map_df["distance_to_cdg_km"] = geo.haversine_km(map_df["latitude"], map_df["longitude"], CDG_COORDS["lat"], CDG_COORDS["lon"])

final_approach_rows = map_df[(map_df["distance_to_cdg_km"] <= MAX_DISTANCE_KM) & (map_df["vertical_rate"] < 0) & (map_df["geo_altitude"] < MAX_ALTITUDE_M)]
arrival_ids = final_approach_rows["icao24"].unique()
arrivals_df = map_df[map_df["icao24"].isin(arrival_ids)]

fig = px.line_map(
	data_frame=arrivals_df,
	lat="latitude",
	lon="longitude",
	hover_name="callsign",
	hover_data=["velocity_kt", "true_track", "geo_altitude_ft"],
	center={
		"lat": 48.86,
		"lon": 2.35
	},
	zoom=8,
	color="callsign",
 	map_style="carto-positron",
  	title=title
)

fig.show()

In [ ]:
# map_df = prepare_map_data(df)

# snapshot_time = pd.to_datetime(data["time"], unit="s", utc=True)

# fig = px.scatter_map(
# 	data_frame=map_df,
# 	lat="latitude",
# 	lon="longitude",
# 	color="geo_altitude_ft",
# 	hover_name="callsign",
# 	hover_data=["velocity_kt", "true_track"],
# 	center={
# 		"lat": 48.86,
# 		"lon": 2.35
#     },
# 	zoom=8,
# 	map_style="carto-positron",
# 	title=f"Aircraft around Paris - {snapshot_time.strftime("%d %b %Y, %H:%M UTC")}"
# )

# fig.show()
# fig.write_html(f"plots/aircraft_paris_{snapshot_time:%Y-%m-%d-%H-%M}.html")